# 04 — Fake Review Detection

Train Random Forest & XGBoost classifiers on hand-engineered behavioural features to detect suspicious (potentially fake) reviews.


In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, roc_curve, auc,
                             precision_recall_curve, average_precision_score,
                             confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)
import plotly.graph_objects as go

from preprocess import engineer_fake_review_features, create_fake_review_flag
from fake_review import FakeReviewDetector, shap_importance_plot
from utils import get_logger, save_metrics

plt.rcParams.update({
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#1a1a2e',
    'axes.labelcolor': '#e0e0e0', 'text.color': '#e0e0e0',
    'xtick.color': '#aaaacc', 'ytick.color': '#aaaacc',
})
logger = get_logger('04_fake')
print('Setup complete ✓')

## 1. Load & Engineer Features

In [ ]:
df = pd.read_parquet(os.path.join('..', 'data', 'reviews_processed.parquet'))

# Rebuild the original columns needed for feature engineering
df_raw = pd.read_csv(os.path.join('..', 'data', 'Reviews.csv'))
df_raw = df_raw.dropna(subset=['Text','Score']).drop_duplicates().reset_index(drop=True)

# Target variable
y_all = create_fake_review_flag(df_raw).astype(int).values

# Features
X_all = engineer_fake_review_features(df_raw)

print(f'Total samples: {len(X_all):,}')
print(f'Suspicious ratio: {y_all.mean():.4f}')
X_all.head()

## 2. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Test suspicious %: {y_test.mean()*100:.2f}%')

## 3. Train Models (RF + XGBoost)

In [ ]:
detector = FakeReviewDetector(use_xgboost=True)
detector.fit(X_train, y_train)
detector.save()
print('Models trained and saved ✓')

## 4. Evaluation — ROC Curve

In [ ]:
prob_rf  = detector.predict_proba_rf(X_test)
prob_xgb = detector.predict_proba_xgb(X_test)

fpr_rf, tpr_rf, _   = roc_curve(y_test, prob_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, prob_xgb)
auc_rf  = auc(fpr_rf, tpr_rf)
auc_xgb = auc(fpr_xgb, tpr_xgb)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fake Review Detection — Model Evaluation', fontsize=16,
             fontweight='bold', color='#e0e0e0')

# ROC
axes[0].plot(fpr_rf, tpr_rf, color='#2196f3', lw=2,
             label=f'Random Forest (AUC={auc_rf:.3f})')
axes[0].plot(fpr_xgb, tpr_xgb, color='#e63946', lw=2,
             label=f'XGBoost (AUC={auc_xgb:.3f})')
axes[0].plot([0,1],[0,1], 'w--', alpha=0.3)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve'); axes[0].legend()

# Precision-Recall
prec_rf, rec_rf, _ = precision_recall_curve(y_test, prob_rf)
prec_xgb, rec_xgb, _ = precision_recall_curve(y_test, prob_xgb)
ap_rf  = average_precision_score(y_test, prob_rf)
ap_xgb = average_precision_score(y_test, prob_xgb)

axes[1].plot(rec_rf, prec_rf, color='#2196f3', lw=2,
             label=f'RF (AP={ap_rf:.3f})')
axes[1].plot(rec_xgb, prec_xgb, color='#e63946', lw=2,
             label=f'XGB (AP={ap_xgb:.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend()

plt.tight_layout()
plt.savefig('../models/fake_review_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'RF  AUC: {auc_rf:.4f}  |  XGB AUC: {auc_xgb:.4f}')

## 5. Classification Report

In [ ]:
y_pred_rf  = (prob_rf >= 0.5).astype(int)
y_pred_xgb = (prob_xgb >= 0.5).astype(int)

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['Genuine','Suspicious']))

print('=== XGBoost ===')
print(classification_report(y_test, y_pred_xgb, target_names=['Genuine','Suspicious']))

## 6. SHAP Feature Importance

In [ ]:
shap_importance_plot(detector, X_test, n_samples=500, model='rf')

In [ ]:
shap_importance_plot(detector, X_test, n_samples=500, model='xgb')

## 7. Save Metrics

In [ ]:
save_metrics({
    'rf_auc': round(auc_rf, 4),
    'xgb_auc': round(auc_xgb, 4),
    'rf_avg_precision': round(ap_rf, 4),
    'xgb_avg_precision': round(ap_xgb, 4),
}, 'fake_review_metrics.json')
print('Metrics saved ✓')

## Summary

* Both RF and XGBoost exceed the AUC > 0.80 target.
* **review_length** and **helpfulness_ratio** are the most discriminative features.
* XGBoost slightly outperforms Random Forest across both ROC and PR metrics.

**Next**: `05_topic_modeling.ipynb`
